In [1]:
import pandas as pd
full = pd.read_csv('FullHeart_Importance.csv')
full = full.sort_values('scores',ascending=True)
small = pd.read_csv('SmallHeart_Importance.csv')
small.index = small.feature
small = small.reindex(full['feature'])

In [2]:
import numpy as np
from scipy.stats import kendalltau, spearmanr, rankdata

# example inputs: dicts {feature: importance}
full  = dict(full.values)
small = dict(small.values)

# align features
features = sorted(full.keys() & small.keys())
x = np.array([full[f]  for f in features])
y = np.array([small[f] for f in features])

# ranks (average ranks for ties)
rx = rankdata(x, method="average")
ry = rankdata(y, method="average")

# Kendall's tau-b (preferred with ties)
tau, p_kendall = kendalltau(rx, ry, variant='b')

# Spearman (also fine)
rho, p_spearman = spearmanr(rx, ry)

print(f"Kendall tau-b={tau:.3f}, p={p_kendall:.3g}")
print(f"Spearman rho={rho:.3f}, p={p_spearman:.3g}")


Kendall tau-b=0.527, p=0.0264
Spearman rho=0.673, p=0.0233


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
mpl.rcParams['font.family'] = 'Helvetica'
# Create figure and axis objects with transparent background
fig, ax = plt.subplots()
fig.patch.set_facecolor('none')
ax.set_facecolor('none')
def normalize_values(values, min_val, max_val):
    return [(x - min_val) / (max_val - min_val) for x in values]

# Sample data (replace with your actual data)
categories = full['feature'].tolist()[-20:]
values_left = full['scores'].tolist()[-20:]
values_right = small['scores'].tolist()[-20:]

# Combine left and right values to find global min and max
all_values = values_left + values_right
min_val = min(all_values)
max_val = max(all_values)

# Normalize the values
normalized_left = normalize_values(values_left, min(values_left), max(values_left))
normalized_right = normalize_values(values_right,min(values_right), max(values_right))
values_left=normalized_left 
values_right=normalized_right
# Combine left and right values to find global min and max
all_values = values_left + values_right
min_val = 0
max_val = 1
# Create a linear segmented colormap
colors = ['#2F3D60', '#4C78A8', '#57BADE', '#9CCCEB', '#D1E0EC', '#E7E8E6'][::-1]
cmap = mcolors.LinearSegmentedColormap.from_list("custom", colors, N=256)
norm = mcolors.Normalize(vmin=min_val, vmax=max_val)

# Create figure and axis objects
# fig, ax = plt.subplots()

# Set the y-axis labels
y_pos = np.arange(len(categories))

# Plot the bars facing left
ax.barh(y_pos, [-x for x in values_left], color=[cmap(x) for x in normalized_left], label='Left Values')

# Plot the bars facing right
ax.barh(y_pos, values_right, color=[cmap(x) for x in normalized_right], label='Right Values')

# Add the category labels to the y-axis
ax.set_yticks(y_pos)
ax.set_yticklabels(categories)

# Add grid, labels, and legend
ax.grid(True)
ax.set_xlabel('Values')
ax.set_title('Side-by-Side Horizontal Bar Plot')

# Add colorbar
cbar = plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, orientation='vertical')
cbar.set_label('Value Scale')

# Customize plot appearance

# Customize plot appearance
ax.spines['left'].set_linewidth(3)    
ax.spines['bottom'].set_linewidth(3)
ax.spines['top'].set_linewidth(3)    
ax.spines['right'].set_linewidth(3)
ax.spines['left'].set_color('black') 
ax.spines['right'].set_color('black') 
ax.tick_params(axis='both', which='major', labelsize=8, length=6, width=3, color='black')

ax.grid(False)
# Show the plot
plt.savefig('HeartImportance.pdf',transparent=True)
